In [44]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

os.environ["USER_AGENT"] = "nomad-research-agent/1.0"

In [45]:
from pathlib import Path

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_openai import ChatOpenAI

In [46]:
wikipedia_search = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=3,
        doc_content_chars_max=2000,
    )
)

duckduckgo_search = DuckDuckGoSearchRun()


@tool
def search_wikipedia(query: str) -> str:
    """
    Search Wikipedia for background information about a topic.
    Use this tool when you need encyclopedic or historical context.
    """
    try:
        return wikipedia_search.invoke(query)
    except Exception as e:
        return f"Wikipedia search failed: {e}"


@tool
def search_duckduckgo(query: str) -> str:
    """
    Search DuckDuckGo for web search results about a topic.
    Use this tool when you need recent articles, websites, or external sources.
    """
    try:
        return duckduckgo_search.invoke(query)
    except Exception as e:
        return f"DuckDuckGo search failed: {e}"

In [47]:
print(search_wikipedia.invoke("XZ backdoor")[:2000])
print("-----")
print(search_duckduckgo.invoke("XZ backdoor")[:2000])

Wikipedia search failed: Expecting value: line 1 column 1 (char 0)
-----
It only triggers when the backdoored xz library gets loaded by a /usr/bin/sshd process on one of the affected distributions. Since the backdoor affects the latest XZ Utils releases, the recommended course of action is to downgrade to an uncompromised release. An intentionally placed backdoor in XZ Utils, an open-source compression utility, was pretty much accidentally discovered by a Microsoft engineer ... Tan spent the year slowly incorporating a backdoor into XZ Utils: disabling systems that might discover his actions, laying the groundwork, and ... ... Fri, 29 Mar 2024 08:51:26 -0700 From: Andres Freund To: oss-security ...ts.openwall.com Subject: backdoor in upstream xz ...


In [48]:
@tool
def scrape_website(url: str) -> str:
    """
    Scrape a website URL and return its text content.
    Use this tool when you need to read the content of a webpage.
    """
    try:
        loader = WebBaseLoader(url)
        docs = loader.load()

        text = "\n\n".join(doc.page_content for doc in docs)

        if len(text) > 8000:
            text = text[:8000]

        return text

    except Exception as e:
        return f"Website scraping failed: {e}"

In [49]:
@tool
def save_research_to_txt(research: str) -> str:
    """
    Save the final research result to a txt file.
    Use this tool only when the research is complete.
    """
    file_path = Path("xz_backdoor_research.txt")

    with open(file_path, "w", encoding="utf-8") as file:
        file.write(research)

    return f"Research saved to {file_path}"

In [50]:
scrape_result = scrape_website.invoke(
    "https://en.wikipedia.org/wiki/XZ_Utils_backdoor"
)

print(scrape_result[:1000])





XZ Utils backdoor - Wikipedia






























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages





		Print/export
	


Download as PDFPrintable version





		In other projects
	


Wikidata item



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools






Donate


Create account


Log in





























Contents
move to sidebar
hide




(Top)





1
Background








2
Mechanism








3
Response




Toggle Response subsection





3.1
Remediation








3.2
Broader response










4
Notes








5
References








6
External links


















Toggle the table of contents







XZ Utils backdoor



14 languages




CatalàČeštinaDeutschEspañolFrança

In [51]:
save_result = save_research_to_txt.invoke(
    "This is a test research result about the XZ backdoor."
)

print(save_result)

file_path = Path("xz_backdoor_research.txt")
print("File exists:", file_path.exists())

Research saved to xz_backdoor_research.txt
File exists: True


In [52]:
tools = [
    search_wikipedia,
    search_duckduckgo,
    scrape_website,
    save_research_to_txt,
]

for tool_item in tools:
    print("Name:", tool_item.name)
    print("Description:", tool_item.description)
    print("-----")

Name: search_wikipedia
Description: Search Wikipedia for background information about a topic.
Use this tool when you need encyclopedic or historical context.
-----
Name: search_duckduckgo
Description: Search DuckDuckGo for web search results about a topic.
Use this tool when you need recent articles, websites, or external sources.
-----
Name: scrape_website
Description: Scrape a website URL and return its text content.
Use this tool when you need to read the content of a webpage.
-----
Name: save_research_to_txt
Description: Save the final research result to a txt file.
Use this tool only when the research is complete.
-----


In [53]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

research_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a professional research AI agent.

Your job is to research a topic using the available tools and then save the final research to a txt file.

You have access to these tools:
- Wikipedia search
- DuckDuckGo search
- Website scraping
- Save research to txt file

Required workflow:
1. First, search Wikipedia for background information.
2. Then, search DuckDuckGo for additional sources.
3. If DuckDuckGo gives you a useful website URL, scrape one relevant website.
4. Write a clear research report.
5. Save the final research report using the save_research_to_txt tool.
6. After the file is saved, immediately give a short final answer and stop.

Research report requirements:
- Explain what the XZ backdoor was.
- Include a short timeline.
- Mention the affected software or project.
- Mention who discovered or reported it if available.
- Explain the security impact.
- Do not make up facts.
- If a fact is uncertain, say that it is uncertain.

Stopping rules:
- Do not call tools forever.
- Do not search repeatedly after you have enough information.
- You must call save_research_to_txt exactly once.
- After save_research_to_txt succeeds, do not call any more tools.
- Your final answer should simply confirm that the research was saved.
""",
)

In [54]:
result = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
Research about the XZ backdoor.

Use Wikipedia, DuckDuckGo, and scrape one relevant website if possible.
When you have enough information, save the final research to a txt file.
After saving the file, stop and tell me that the research was saved.
""",
            }
        ]
    },
    config={
        "recursion_limit": 30,
    },
)

print(result["messages"][-1].content)

The research was saved successfully.


In [55]:
file_path = Path("xz_backdoor_research.txt")

print("File exists:", file_path.exists())
print("File path:", file_path.resolve())

File exists: True
File path: /Users/browndays/VSCode/langchain/xz_backdoor_research.txt
